# [0] Code Organization
The files submitted for this project are organized in the following manner:

-- `main.py`: Top-level file for running the entire robot system.

-- `constants.py`: Global constants shared between `main` and all modules.

-- `test.py`: File containing a collection of tests that were used to characterize the robot and system.

-- `classes/`: Contains all code modules.

------ `cv/`: Files for the computer vision module.

------ `gp/`: Files for the global planning module.

------ `ekf.py`: Contains the extended Kalman filter module.

------ `local_avoidance.py`: Contains the local navigation module.

------ `motion_control.py`: Contains the motion control (PID) module.

-- `data collection/`: Excel files containing data used in robot/CV characterization.

-- `main.ipynb`: This report.

-- `images/`: Images for this report.

-- `requirements.txt`: Python modules to install in order to run the system.

# [1] Main File
The top-level file for running the entire robot system is `main.py`. This file calls all of the other sub-modules and manages the entire process, from vision initialization to stopping at the end goal. This file contains the function `main()`, which is what is directly run, and many other helper functions to manage all specific actions required.

`main()` first runs all required initialization steps (setting up the robot, computer vision, and global path), and then enters a state machine for the remainder of the program. The possible states are:

- `GLOBAL_NAV`: Normal global navigation behaviour: The robot should be following the generated path, using the filtering and motion control modules to stay on the path.
- `LOCAL_NAV`: Local obstacle avoidance: The robot has sensed an obstacle and must stray from its ideal trajectory to avoid a collision.
- `KIDNAPPED`: The robot has been kidnapped. It is currently travelling through the air, and should no longer be attempting to follow any path.
- `WAIT`: This state runs after a kidnapping has been completed. The robot has a slight delay after being set down, before it recalculates the global path and continues towards the goal.
- `STOP`: The robot has either been forced to stop, or it has reached its final goal. Either way, the program should now end.

The program therefore has three main phases: initialization, main loop, and conclusion.

## [1A] Initialization
The high-level initialization steps performed are as follows:

1. Initialize robot (`init_tdm()`)
  - Wait for connection to robot 
  - Wait for variables to populate
  - Add a variables changed listener (`variables_changed()`). This function ensures that the robot will enter the `STOP` state whe middle button is pressed, aborting the program.
2. Initialize computer vision (`init_vision()`)
  - Show window
  - Wait for a press of the 'h' key, then compute locations of corner tags, homography matrix, and locations of obstacles
  - Wait for a press of the 'esc' key to confirm
  - Get robot start and goal positions
  - Convert start position to robot coordinate frame and log
3. Initialize global path (`init_global_path()`, called at the end of `init_vision()`)
  - Call the global planner module to calculate the visibility graph and A* path for the robot to follow
  - Convert path data to robot coordinate frame
4. Initialize control classes (`EKF`, `PID`, `LocalNav`) and set starting variables (goals, state log, etc.)

## [1B] Main Loop
After initialization, the robot runs the following steps on repeat until it enters the `STOP` state.

1. Update live feed
2. Check kidnapping conditions (has the robot newly been picked up/set down)
  - (Note: Kidnapping stops all other functions, and therefore overrides the other states)
  - If it has just been lifted, switch to `KIDNAPPED` state
  - If it has just been set down, switch to `WAIT` state
3. Act appropriately based on current kidnapped state (if applicable)
  - `KIDNAPPED`: Just set the motors to 0
  - `WAIT`: Check if it has completed its wait time. If it has, re-run the following initialization steps to refresh the robot and get a new global path (`wait_after_kidnapping()`):
    - Get new robot pose
    - Update vision to get new obstacles (just in case anything changed)
    - Get a new global path
    - Re-initialize all control classes
4. If not kidnapped, get the current pose and check if we've reached the goal point within the allowed buffer
  - `get_current_pose()` returns the current robot state estimate, after applying an update with the EKF. It does the following:
    - Get the current robot wheel speeds and convert to mm/s
    - Get the current robot position from computer vision
    - If the robot tag isn't currently visible, update the EKF with a very large uncertainty on measurements (increasing the values in the `R` matrix)
    - If the tag is newly visible, update `R` back to its normal values
    - Get the next state from the EKF, using wheel speeds as inputs and CV measurements as measurements
  - `reached_goal()` simply compares the current and goal poses, and returns True/False depending on whether we're within the allowed buffer zone
    - If we've reached an intermediate waypoint, update the goal to the next waypoint in the list
    - If this is the last waypoint (a.k.a. the end goal), set the state to `STOP` to conclude the program
5. Check for any new obstacles detected by the front proximity sensors (`obstacle_detected()`). If so, set state to `LOCAL_NAV`.
6. Act based on navigation state:
  - `GLOBAL_NAV`: Sends the current and goal poses to the PID controller to get the desired motor velocities, then sets the motors.
  - `LOCAL_NAV`: Employs the local navigation edge following process to travel around the obstacle
7. Sleep for any extra time. 
  - The loop runs at a set time interval (`constants.general['dt']`) to ensure accurate state estimation updates. The processes within the loop have variable runtimes, so `dt` is set to slightly higher than the maximum typical runtime, and the robot sleeps for the remaining time at the end of each loop.

## [1C] Conclusion
The main loop ends when the state is set to `STOP`, either by pressing the center button or reaching the end goal. At this point, the motors are shut off, the vision is closed, and the state and error plots are shown for analysis.


## [3] Extended Kalman Filter
### [3A] Module Overview
An Extended Kalman Filter was implemented for state estimation of the robot. This filter combines robot position (x, y, and angle) measurements from the computer vision with a basic two-wheel differential robot motion model for state estimation, using motor speeds produced from the PID controller (next section) as inputs.

The code for the EKF is located in `classes/ekf.py`. To use it, an instance of the `EKF` class must be initialized with the following parameters:
- Motion model error covarince matrix `Q`
- Measurement model error covariance matrix `R`
- Initial state
- Robot wheel-to-wheel width (used in the motion model) `T`
- Time step `dt`

After initialization, the next state can be calculated by calling the function `generate_next_state(inputs, measures)`, which takes as parameters the system inputs (motor velocities) and measurements (positions from CV) at the current time.

### [3B] Independent Showcase
The following code blocks provide a simple simulation demonstrating the functionality of the EKF. At each time step, the robot is provided with a random input. Its next state is moved according to the motion model and measurements are generated at each step, with noise added based on the measured variances (see next subsection).

We first import the required modules, select the starting pose of the robot, and initialize the EKF.

In [ ]:
from classes.ekf import EKF
import constants
import numpy as np
import matplotlib.pyplot as plt
import random

# Set start at the origin
# State takes the form [x, y, v, theta, w]
start_pose = [0., 0., 0., 0., 0.]
dt = constants.general['dt']

# Initialize EKF
ekf = EKF(
    np.diag(constants.ekf['move_vars']), # Q
    np.diag(constants.ekf['measure_vars']), # R
    start_pose,
    constants.robot['wheel_base'],
    dt
)

We can then run the entire loop, which takes the following steps:
- Get the ideal measurements (the exact current state), then add noise based on the proper measurement variance
- Get the state estimate from the EKF
- Pick a set of random inputs to apply
- Update the real state using these inputs, and add noise based on the proper motion model variance

In [ ]:
# Logs for plotting later
real_pose_log = [start_pose]
ekf_pose_log = []

# Initial inputs are 0
left_mtr = 0
right_mtr = 0

iters = 100
for n in range(iters):
    prev_pose = real_pose_log[-1]

    # Get ideal measurements from current state (x, y, theta)
    measures_ideal = [prev_pose[0], prev_pose[1], prev_pose[3]]

    # Add noise based on measured variances (constants.ekf['measure_vars'])
    measures_noisy = measures_ideal
    for i in range(len(measures_noisy)):
        measures_noisy[i] += random.gauss(0, np.sqrt(constants.ekf['measure_vars'][i]))

    # Get state estimate from EKF
    est_pose = ekf.generate_next_state(
                inputs=[left_mtr, right_mtr],
                measures=measures_noisy
                )
    ekf_pose_log.append(est_pose)

    # Break if we're at the last step, we don't care about the last position
    if n == iters-1:
        break

    # Pick new random inputs based on our normal driving velocity ranges
    left_mtr = random.randrange(50, 150)
    right_mtr = random.randrange(50, 150)

    # Calculate linear and angular velocity for simulation purposes
    v = (left_mtr + right_mtr)/2
    w = (right_mtr - left_mtr)/constants.robot['wheel_base']

    # Calculate theoretical next state
    prev_x, prev_y, prev_v, prev_theta, prev_w = real_pose_log[-1]

    next_x = prev_x + v*np.cos(prev_theta)*dt
    next_y = prev_y + v*np.sin(prev_theta)*dt
    next_theta = prev_theta + w*dt

    real_pose_nonoise = [next_x, next_y, v, next_theta, w]

    # Add noise
    real_pose = real_pose_nonoise
    for i in range(len(real_pose)):
        real_pose[i] += random.gauss(0, np.sqrt(constants.ekf['move_vars'][i]))

    real_pose_log.append(real_pose)



Finally, we plot the real and estimated state.

In [ ]:
# Plot path
x_log_real = [state[0] for state in real_pose_log] 
y_log_real = [state[1] for state in real_pose_log]
theta_log_real = [state[3] for state in real_pose_log]

x_log_ekf = [state[0] for state in ekf_pose_log] 
y_log_ekf = [state[1] for state in ekf_pose_log]
theta_log_ekf = [state[3] for state in ekf_pose_log]

plt.subplot(2, 1, 1)
plt.plot(x_log_real, y_log_real)
plt.plot(x_log_ekf, y_log_ekf)
plt.title("Robot path travelled")
plt.legend(["Real path", "EKF estimate"])
plt.axis('equal')

plt.subplot(2, 1, 2)
plt.plot(theta_log_real)
plt.plot(theta_log_ekf)
plt.legend(["Real path", "EKF estimate"])
plt.title("Theta log")
plt.show()

### [3C] Deep Dive & Explanation
#### [3C-1] Filter Selection
The three types of filtering for localization taught in class were the histogram filter, particle filter, and Kalman filter. Of these three, we decided that a Kalman filter would be the best fit for our scenario. 

Using a histogram filter would have involved discretizing the entire map into a fine enough grid to provide high-accuracy localization - on the order of millimeters. Splitting our map of size 890 x 1260 mm into even 5 mm square sections would require storing and recalculating $(890/5)*(1260/5) = 44856$ different probabilities each timestep, using a large amount of processing power for what would still be an inaccurate state estimate (and this is only including x and y position as state variables). This was clearly not an ideal solution.

A particle filter is a lower-cost version of the histogram filter, prioritizing likelihood calculations for only the most probable states. However, this still has a much higher computational cost than a Kalman filter, and likely a lower accuracy (depending on the size of the discretization step). 

In terms of computing power and accuracy, a Kalman filter is the best obvious choice. There are two main concerns with using a Kalman filter. First of all, the generic Kalman filter can only work with linear systems, so an Extended Kalman Filter (EKF) would need to be used instead. This is slightly more complex, as it requires calculating Jacobians of the motion and measurement model matrices, and would lose accuracy on highly nonlinear systems. We assumed, however, that the two-wheel differential drive equations of motion are simple enough that an EKF would still perform well (i.e. these equations are not "highly nonlinear").

The other, more important drawback of using an EKF is that it assumes the measurement and motion model errors follow a Gaussian distribution. Before the EKF could be reliably implemented and used, it had to be shown that this assumption was true.

#### [3C-2] Motion and Measurement Models
The Thymio robot is a two-wheel differential drive robot. If we know the linear speeds of each wheel at a given timestep $k$ ($v^l_k$ and $v^l_k$ for the left and right wheels, respectively), and that the robot has a distance of $T$ between the two wheels, then the velocity and angular velocities of the robot in the global frame are:

$$v_k = \frac{v^l_k + v^r_k}{2} \quad \omega = \frac{v^r_k - v^l_k}{T}$$

Using a first-order Taylor series approximation, the $x$, $y$, and $\theta$ states of the robot can therefore be updated from the previous state using the following equations:

$$x_k = x_{k-1} + v_{k-1} cos(\theta_{k-1})dt$$
$$y_k = y_{k-1} + v_{k-1} sin(\theta_{k-1})dt$$
$$\theta_k = \theta_{k-1} + \omega_{k-1} dt$$

We use the state variable $\mu = [x, y, v, \theta, \omega]^T$. Therefore, the measurement model function $g(\mu_{k-1}, u_t)$ (where $\mu_{k-1}$ is the previous state and $u_t$ is the current input) is:

$$ g(\mu_{k-1}, u_t) = \begin{bmatrix}
x_{k-1} + v_{k-1} cos(\theta_{k-1})dt \\
y_{k-1} + v_{k-1} sin(\theta_{k-1})dt \\
\frac{v^l_k + v^r_k}{2} \\ 
\theta_{k-1} + \omega_{k-1} dt \\
\frac{v^r_k - v^l_k}{T}
\end{bmatrix}

= \begin{bmatrix}
\mu^{(0)}_{k-1} + \mu^{(2)}_{k-1} cos(\mu^{(3)}_{k-1})dt \\
\mu^{(1)}_{k-1} + \mu^{(2)}_{k-1} sin(\mu^{(3)}_{k-1})dt \\
\frac{u^{(0)}_k + u^{(1)}_k}{2} \\ 
\mu^{(3)}_{k-1}+ \mu^{(4)}_{k-1} dt \\
\frac{u^{(1)}_k - u^{(0)}_k}{T}
\end{bmatrix} $$

The Jacobian $G(\mu_{k-1})$ contains the partial derivatives of each function in $g$ with respect to each state variable in $\mu$. This is therefore:

$$
G(\mu_{k-1}) = \begin{bmatrix}
\frac{dg_1}{d\mu} \\
\frac{dg_2}{d\mu} \\
\frac{dg_3}{d\mu} \\
\frac{dg_4}{d\mu} \\
\frac{dg_5}{d\mu} \\
\end{bmatrix}
= 
\begin{bmatrix}
1 & 0 & cos(\theta_{k-1})dt & -v_{k-1}sin(\theta_{k-1})dt & 0 \\
0 & 1 & sin(\theta_{k-1})dt & v_{k-1}cos(\theta_{k-1})dt & 0 \\
0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 1 & dt \\
0 & 0 & 0 & 0 & 0
\end{bmatrix}

$$

The measurement model is much simpler. We measure $x$, $y$, and $\theta$ directly from the computer vision, therefore the measurement model $h(\mu_{k-1})$ is simply:

$$
h(\mu_{k-1}) = \begin{bmatrix}
x_{k-1} \\ y_{k-1} \\ \theta_{k-1}
\end{bmatrix}
= 
\begin{bmatrix}
\mu^{(0)}_{k-1} \\ \mu^{(1)}_{k-1} \\ \mu^{(3)}_{k-1}
\end{bmatrix}
$$

The Jacobian is therefore:

$$
H(\mu_{k-1}) = \begin{bmatrix}
\frac{dh_1}{d\mu} \\
\frac{dh_2}{d\mu} \\
\frac{dh_3}{d\mu}
\end{bmatrix}
=
\begin{bmatrix}
1 & 0 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 & 0 \\
0 & 0 & 0 & 1 & 0
\end{bmatrix}
$$

#### [3C-3] Characterization
Before applying the EKF equations above, we had to characterize the variances of each equation in the motion and measurement models, and confirm the errors formed Gaussian distributions. We also had to determine a conversion between linear speed in mm/s, and motor speed in Thymio units. 

We first tested the error magnitude and variance of the computer vision position estimates, to determine if they were accurate enough to be used as a "ground truth" for robot characterization. For x and y position, we placed a point at a known location in the center of the map. The CV estimate should be most accurate there, as it is directly below the camera and undergoes the least distortion. We then took 216 repeated measurements. (Note: We decided to take all of our measurements at one location, because we observed in earlier tests that the error in our manual measurements was creating a multimodal distribution due to different constant errors at each location).

To measure the angular error, we again centred a tag, then took several measurements at each angle in between 0 and 360 degrees (with steps of 20 degrees). We repeated this at three different locations spread across the map.

This data is available in `data collection/cv_variances.xlsx`. The images below show the distributions of these three error measurements. They are each roughly Gaussian, which is the first step in validating the choice of EKF for localization. The variances were:
- x: 0.364 mm
- y: 0.213 mm
- $\theta$: 0.000505 rad

giving a measurement covariance matrix of 
$$
R = \begin{bmatrix}
0.364 & 0 & 0 \\
0 & 0.213 & 0 \\
0 & 0 & 0.00051
\end{bmatrix}
$$

Note that it was assumed these variables are each independent. 

![](images/cv_errors.png)

Next, we ran a test to determine the conversion between linear speed in mm/s and speed in Thymio units. We decided to use the computer vision as a measurement model, as based on the previous error variance results, we estimated that this was much more accurate than what we could achieve manually.

All of our robot characterization tests are located in `tests.py`. The function used for this test is `test_velocity_to_motor_input()`. This test follows the following steps:
1. Initialize vision
2. Set a list of robot speeds to test, and a runtime at each speed
3. Set the motors to the first speed (in Thymio units)
4. Wait 1 second to account for acceleration
5. Store the current robot position and timestamp
6. Wait for the specified delay time
7. Store the end robot position and new timestamp
8. Turn off the motors
9. Calculate the distance travelled using the CV positions, then calculate real-world velocity using this distance and the stored timestamps
10. Log the pair of speeds (Thymio units & mm/s)

We first calibrated the Thymio to ensure it drove straight, using the process described [in the documentation](http://aseba.wikidot.com/fr:thymiomotorcalibration). This calibration mode allows the user to adjust the left and right gains for the two motors by pressing the two side buttons on the top of the robot. As the robot moved, we adjusted the gains until it drove straight, verifying this calibration at several different speeds.

After calibration, we ran our test function several times at a variety of different speeds, then performed linear regression in Excel to determine a conversion. The data is located in `data collection/motor-speed-calibration.xlsx`. This produced the following plot, and the relationship `real_speed = 0.3664*thymio_speed` with an $R^2=0.9994$ (nearly a perfect fit).

![](images/mtr_speed_calibration.png)

Finally, we ran a series of characterization tests to determine the variances of the five state variables from the motion model. These tests are located in the `test_linear_variances()` and `test_angular_variances()` functions of `test.py`. 

For the linear variances, the steps followed were:
1. Initialize vision
2. Start robot motors
3. Wait 1 second to account for acceleration
4. Log the current time and robot pose, then add a slight delay
5. At each timestep for the specified test time:
6. Log the current timestep and robot pose
7. Get the robot's two wheel speeds, and calculate velocity from the average of the two
8. Calculate velocity error as the difference between the set and measured velocities
9. Calculate two different estimates of position, one from CV (as a "ground truth") and one from robot odometry (imitating the EKF motion model). CV calculates the Euler distance to the previous pose, and the robot uses $velocity*\Delta t$. 
10. Calculate the positional error as the difference between these two, and log.

The steps followed for angular variances were nearly the same. The two robot wheels were set to different speeds to produce a known (theoretical) angular velocity, and the absolute angle $\theta$ of the robot was measured from CV instead of the distance travelled.

These tests gave four variances: linear and angular positions and velocities. There are five state variables, as linear position has an $x$ and a $y$ component. It was assume that the variances for both of these components would be the same, as the robot should theoretically perform the same regardless of which direction it is facing.

The data acquired from these tests is provided in `data collection/robot-variances.xlsx`. It produced the plots showed below. As we can see, each plot follows a roughly Gaussian distribution. This confirms that everything being modelled (measurements and motion) follow a Gaussian error distribution, therefore validating the choice of an EKF.

The variances of each measurement are:
- Position: 4.48 mm
- Velocity: 10.78 mm/s
- Angle: 0.00155 rad
- Angular velocity: 0.00878 rad/s

For position and angle, we must account for the error in the CV "ground truth" as well. We therefore add the CV x, y, and $\theta$ variances to these values, to produce

$$
Q = \begin{bmatrix}
4.844 & 0 & 0 & 0 & 0 \\
0 & 4.693 & 0 & 0 & 0 \\
0 & 0 & 10.78 & 0 & 0 \\
0 & 0 & 0 & 0.00207 & 0 \\
0 & 0 & 0 & 0 & 0.00878
\end{bmatrix}
$$

Note that we are again assuming that each variable is independent for simplicity.

![](images/robot_variances.png)

#### [3C-4] Implementation
With the selected motion and measurement models, and the data collection displayed above, we have successfully shown that all relevant variables in the EKF follow a Gaussian distribution, therefore validating this choice. The measured variances were stored in `constants.py` and the motion model, measurement model, and Jacobians were implemented in `classes/ekf.py`.

To update the state, the `EKF.generate_next_state()` function is used, taking CV x, y, $\theta$ readings as measurements and the two wheel speeds (converted to mm/s) as inputs. The function follows the following steps (standard for an Extended Kalman Filter):
1. Calculate the belief of the next state using the motion model.
2. Calculate the belief of the next covariance matrix (P) using the motion model Jacobians, motion model covariance, and previous covariance.
3. Calculate the new Kalman gain using the measurement model Jacobian, measurement model covariance matrix, and covariance matrix belief.
4. Update the estimated state from the belief by calculating the error between the actual and modelled measurements, then scaling it by the new Kalman gain.
5. Update the estimated covariance using the Kalman gain and measurement Jacobian.

When the EKF is first initialized P is set equal to Q (the motion model covariance) as this is a reasonable "worst-case scenario" for this covariance matrix.

*Disclaimer: My notes from a previous course [(MTE 544 at the University of Waterloo)](https://ucalendar.uwaterloo.ca/2223/COURSE/course-MTE.html#:~:text=MTE%20544%20LAB%2CLEC%2CTUT%200.50) were consulted to acquire further details on the Extended Kalman Filter, as it was only briefly discussed during this course.*

#### [3C-5] Example
The following images show four screenshots of the robot running along an example path, and the effect of the EKF on the position estimate. The target path is plotted as blue lines, and the estimated state at each timestep is plotted as a red dot. The blue ellipses in the second and third images show the uncertainty of the state estimate (one standard deviation) from the most recently calculated EKF covariance matrix. 

These ellipses are theoretically plotted in the first and fourth images as well, however they are so small (due to the high accuracy of the computer vision pose estimation) that they are invisible. When the camera is covered in the second and third images, the measurement model covariance is set to near-infinity, meaning that the EKF is now relying only on the less accurate measurement model and there is a large uncertainty of the position. 

The ellipse has a larger radius perpendicular to the robot's travel direction. This is because two wheel differential drive robots such as the Thymio are much more accurate in driving straight than they are in turning. A slight error in angle will have more impact on the final position than a slight error in linear position, as the angular error will cause the linear error to accumulate more and more over time. We can see the ellipse growing as the robot progresses further with the covered camera, and becomes less and less confident in its global position. When the camera is uncovered and we regain our exteroceptive sensor, this uncertainty goes back down again. It does not accumulate when the camera is present, as the camera measures the global position directly, instead of estimating it from every previous global position estimate.

![](images/camera-cover-ekf.png)

## [4] Motion Control
### [4A] Module Overview
To allow the robot to smoothly and accurately follow the desired path, a PID controller was implemented. This controller is located in the `PID` class in `classes/motion_control.py`.

To use the PID controller, an instance of the `PID` class must first be created. Then, the `get_motor_vels(start, goal)` function can be called every time robot's motion requires an update, where `start` and `goal` take the form of a pose in the form `[x, y, theta]`. The class stores previous error values in order to determine the error integrals and derivatives, and provides plotting capability. The PID constants used are stored in the dictionary `pid` in `constants.py`. 

### [4B] Independent showcase
The following code provides a simple simulation demonstrating the PID controller at work. The robot starts facing vertically, and must make a 90 degree turn to end up at a point to the right of where it began, facing horizontally.

We first import the required modules and initialize the PID class, passing it the desired PID constants.

In [ ]:
import constants
from classes.motion_control import PID
import matplotlib.pyplot as plt
import numpy as np

# Select PID constants
pid_consts = {
    'theta': {  
        'kp': 10,
        'ki': 0,
        'kd': 0, 
    },
    'beta': {
        'kp': 80,
        'ki': 10,
        'kd': 0,
    }
}

# Initialize PID class
pid = PID(pid_consts)

Then, we can set sample data. The starting point is set to the origin, with the robot at an angle of 90 degrees (facing upwards). The end goal is set to a point to the right of this, with an angle of 0. The robot will have to make a sharp turn to and then proceed towards the end goal.

In [ ]:
# Sample data
start = [0, 0, 0, np.pi/2, 0] # State stores [x, y, v, theta, w]
end = [500, 0, 0]
current_pose = start
dt = 0.2

Next, we define a helper function to determine whether or not we have reached our end goal, within a specified tolerance. Note that angle is ignored here - we only care whether we have reached the correct (x, y) position.

In [ ]:
# Use this function to determine whether we've reached our goal,
# wtihin the allowed buffer distance. Note that we ignore angle
# for this comparison.
def reached_goal(current, goal):
    px, py = current[0], current[1]
    gx, gy = goal[0], goal[1]

    # Check that both distances are less than the buffer
    return (abs(px - gx) < constants.general['goal_buffer'] and
        abs(py - gy) < constants.general['goal_buffer'])

We can then run the main simulation loop, using the PID class to get updated motor velocities at each time step.

In [ ]:
# Log the robot's state at each timestep
state_log = [start]

max_iters = 1000
i = 0
# Run main loop
while not reached_goal(current_pose, end):
    if i > max_iters:
        print("Could not reach goal")
        break
    # Get motor velocities from PID. This would be input to the robot.
    mtr_left, mtr_right = pid.get_mtr_vels(current_pose, end)

    # Calculate linear and angular velocity for simulation purposes
    v = (mtr_left + mtr_right)/2
    w = (mtr_right - mtr_left)/constants.robot['wheel_base']

    # Update state for simulation purposes
    prev_x, prev_y, prev_v, prev_theta, prev_w = state_log[-1]

    next_x = prev_x + v*np.cos(prev_theta)*dt
    next_y = prev_y + v*np.sin(prev_theta)*dt
    next_theta = prev_theta + w*dt

    current_pose = [next_x, next_y, v, next_theta, w]
    state_log.append(current_pose)
    i += 1

Finally, we generate plots of the robot's path travelled, and informative PID logs.

In [ ]:
# Plot path
x_log = [state[0] for state in state_log] 
y_log = [state[1] for state in state_log]
theta_log = [state[3] for state in state_log]

plt.subplot(2, 1, 1)
plt.plot([start[0], end[0]], [start[1], end[1]])
plt.plot(x_log, y_log)
plt.title("Robot path travelled")
plt.legend(["Desired path", "Actual path"])
plt.axis('equal')

plt.subplot(2, 1, 2)
plt.plot(theta_log)
plt.title("Theta log")
plt.show()

# Plot PID error logs
pid.plot()

### [4C] Deep Dive and Explanation
#### [4C-1] Choice of Controller
Originally, we had used the Astolfi controller mentioned in class for motion control. We chose this as it was a demonstratedly stable waypoint tracking system with a fairly simple implementation and few constants to tune. However, we observed some performance issues with our implementation of the controller, despite tuning. Some reasons for this were:
- The theoretical Astolfi controller is for a continuous system. We were required to discretize it, and with a fairly slow update frequency (~5 Hz due to motor and CV update times), it became much more difficult to tune the controller to achieve stable and well-performing results.
- The Astolfi controller slows down the velocity of the robot until it comes to a smooth stop at the waypoint. Our path had multiple waypoints, and we wanted to follow it smoothly to the end insead of stopping at each one. We therefore normalized the velocities that were output from the controller to ensure that the robot always travelled at a constant speed. This, however, caused the system to further differ from the expected Astolfi behaviour and caused performance issues such as instability and oscillations.

Due to these issues, we decided to pivot from the base Astolfi controller, and implement a PID controller based on the concepts used in the Astolfi controller. With the normalized velocity, our Astolfi controller had become just a proportional controller, and we hypothesized that the addition of integral and/or derivative terms would help us reduce oscillations and improve path tracking.

#### [4C-2] Controller Implementation
The PID controller looks at two separate angle variables: $\theta$, which represents angle the robot is facing compared to the desired angle at the waypoint, and $\beta$, which represents the angle the robot is facing compared to the angle of the straight line to the goal point. These are the same as the variables $\beta$ and $\alpha$ in the Astolfi controller, simply renamed. 

The controller is therefore managing six different values: the proportional, integral, and derivative errors of both of these variables. The proportional error is simply the differnce between the current heading and the variable being compared (angle of waypoint for $\theta$, or angle to goal for $\beta$). The two integral errors the sum of all previous values of each respective error measurement, and the derivative is the difference between the current error measurement and the previous one. 

The left and right motor velocities are initially set to the same value, our desired normal driving speed. Then, they are each updated (adding to the left, and subtracting from the right) based on the six error calculations - each error parameter is multiplied by its respective PID constant. Finally, these motor values are saturated at a set maximum velocity, to ensure the robot does not go too fast across the small map.

#### [4C-3] Tuning
We began the tuning process using the $\beta$ and $\alpha$ constants from the example Astolfi controller online, as a starting reference. The robot took too long to rejoin the path, so we increased the constant for our $\beta$ (Astolfi $\alpha$) until the robot began to join the path quickly, but with some overshoot. We then added a small derivative constant on $\beta$ to correct for this overshoot. At this point, the robot was performing well, but often had a steady state error - travelling at the correct angle, but slightly offset from the line. We therefore added an integral constant to $\beta$ as well to ensure that, as this small steady-state error accumulated, the robot would be inclined to turn back towards the path.

### [4D] Example Results
The following two images show the comparison between the robot's ideal path, actual (estimated) path, PID constants, and motor inputs for an example path. It can be seen that the robot turns more closely towards the path as the integral sums increase, indicating an improvement over the base proportional controller. Note that integral sums are manually reset to 0 once the robot reaches each waypoint.

![](images/robot-path-example.png) ![](images/pid-output-example.png)